# Multi-Agent · Day 37 — **Agent Reliability & Fault Tolerance**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~2 hrs

A single agent that fails once is an inconvenience. A four-agent pipeline where any hop can fail is a **reliability engineering** problem — and it's the part of multi-agent systems that separates a demo from something you'd run against a bank's loan book. Three tools cover almost all of it: **retry with backoff** (transient failures), the **circuit breaker** (stop hammering a dead dependency), and **graceful partial failure** (return a degraded-but-honest result instead of crashing).

Everything below is deterministic and runnable — we simulate Bedrock throttling and a dead database with counters, so the retry and breaker logic is *observable*, not hand-waved. (Today's Python notebook does the same with the real `tenacity` library.)

Today:
1. **Retry with exponential backoff + jitter** — the correct answer to a 429.
2. The **circuit breaker** — three states, and why "fail fast" beats "retry forever".
3. **Graceful partial failure** — degrade a pipeline honestly.
4. The **runbook** — what a human does when the orchestrator itself dies.

## 1. Retry with exponential backoff + jitter

A `429 ThrottlingException` from Bedrock is *transient* — retrying after a wait usually works. But retry naively and 100 agents all retry at the same instant (a "thundering herd"). Exponential backoff spaces attempts out; jitter de-synchronises them.

In [1]:
import random, time

class ThrottleError(Exception):
    """Stands in for botocore ThrottlingException / HTTP 429."""

def flaky_bedrock(_call={"n": 0}):
    """Fails the first 2 times, succeeds on the 3rd — deterministic for the demo."""
    _call["n"] += 1
    if _call["n"] < 3:
        raise ThrottleError(f"429 (attempt {_call['n']})")
    return {"text": "risk grade BBB", "attempt": _call["n"]}

def retry_with_backoff(fn, max_attempts=5, base=0.05, cap=1.0):
    for attempt in range(1, max_attempts + 1):
        try:
            return fn()
        except ThrottleError as e:
            if attempt == max_attempts:
                raise
            # exponential: base*2^(n-1), capped, plus full jitter
            delay = min(cap, base * 2 ** (attempt - 1))
            delay = random.uniform(0, delay)           # full jitter
            print(f"  {e} -> backing off {delay*1000:.0f}ms")
            time.sleep(delay)

print("result:", retry_with_backoff(flaky_bedrock))

  429 (attempt 1) -> backing off 3ms
  429 (attempt 2) -> backing off 16ms
result: {'text': 'risk grade BBB', 'attempt': 3}


☝ Two non-negotiables. **Only retry transient errors** — a 429 or a timeout, yes; a 400 "malformed request" or a validation error, never (retrying a deterministic failure just wastes time and money). **Always cap attempts** — an unbounded retry loop is how a transient blip becomes an outage. Full jitter (`uniform(0, delay)`) is what stops every retrying agent from waking at the same millisecond and re-throttling the service — counter-intuitively, *more* randomness gives *smoother* recovery.

## 2. The circuit breaker

Retry assumes the dependency will recover *soon*. If it's genuinely down, retrying just burns latency and load. The circuit breaker watches the failure rate and, once it trips, **fails fast** — rejecting calls instantly for a cooldown — then tests the water with a single probe.

In [2]:
from enum import StrEnum

class State(StrEnum):
    CLOSED = "closed"        # healthy: calls pass through
    OPEN = "open"            # tripped: reject instantly, don't even try
    HALF_OPEN = "half_open"  # probing: allow one test call

class CircuitBreaker:
    def __init__(self, fail_threshold=5, cooldown=2.0):
        self.fail_threshold, self.cooldown = fail_threshold, cooldown
        self.failures = 0
        self.state = State.CLOSED
        self.opened_at = 0.0

    def call(self, fn):
        if self.state == State.OPEN:
            if time.time() - self.opened_at >= self.cooldown:
                self.state = State.HALF_OPEN                 # time to probe
            else:
                raise RuntimeError("circuit OPEN — failing fast")
        try:
            result = fn()
        except Exception:
            self.failures += 1
            if self.failures >= self.fail_threshold or self.state == State.HALF_OPEN:
                self.state, self.opened_at = State.OPEN, time.time()
            raise
        # success: probe passed (or healthy) -> reset
        self.failures, self.state = 0, State.CLOSED
        return result

def dead_database():
    raise ConnectionError("db unreachable")

cb = CircuitBreaker(fail_threshold=3, cooldown=1.0)
for i in range(6):
    try:
        cb.call(dead_database)
    except Exception as e:
        print(f"call {i}: state={cb.state:9} -> {type(e).__name__}: {e}")

call 0: state=closed    -> ConnectionError: db unreachable
call 1: state=closed    -> ConnectionError: db unreachable
call 2: state=open      -> ConnectionError: db unreachable
call 3: state=open      -> RuntimeError: circuit OPEN — failing fast
call 4: state=open      -> RuntimeError: circuit OPEN — failing fast
call 5: state=open      -> RuntimeError: circuit OPEN — failing fast


☝ Watch the transition: after 3 real failures the breaker flips to **OPEN**, and calls 3–5 are rejected *instantly* with "failing fast" — no 30-second timeout, no load on the dying database. That's the whole value: a healthy fallback path (call Tool B, return cached data, degrade) can run in milliseconds instead of waiting on a dependency that isn't coming back. After `cooldown` the breaker goes **HALF_OPEN** and lets exactly one probe through; success closes it, failure re-opens it. Retry and breaker are complementary — **retry** for a blip, **breaker** for an outage.

## 3. Graceful partial failure

In a 4-agent pipeline, if the *enrichment* agent dies, you don't crash the memo — you produce it flagged "enrichment unavailable". A degraded honest answer beats a 500. Model each hop as returning a result **or** a stated degradation.

In [3]:
from dataclasses import dataclass, field

@dataclass
class StepResult:
    step: str
    ok: bool
    data: dict = field(default_factory=dict)
    degraded_reason: str = ""

def run_step(name: str, fn, *, required: bool) -> StepResult:
    try:
        return StepResult(name, ok=True, data=fn())
    except Exception as e:
        if required:
            raise                                    # e.g. no customer id -> cannot proceed
        return StepResult(name, ok=False, degraded_reason=str(e))   # optional -> degrade

pipeline = [
    ("ingest",  lambda: {"customer": "C-4417"},                 True),
    ("enrich",  lambda: (_ for _ in ()).throw(TimeoutError("KB timeout")), False),
    ("score",   lambda: {"grade": "BBB"},                       True),
]
results = [run_step(n, f, required=req) for n, f, req in pipeline]

confidence = sum(r.ok for r in results) / len(results)
for r in results:
    print(f"  {r.step:8} {'OK' if r.ok else 'DEGRADED: ' + r.degraded_reason}")
print(f"\nmemo confidence: {confidence:.0%} — "
      f"{'ship with caveat' if confidence >= 0.6 else 'block for review'}")

  ingest   OK
  enrich   DEGRADED: KB timeout
  score    OK

memo confidence: 67% — ship with caveat


☝ The distinction that matters in a bank: **required vs optional** steps. A missing customer id is a *required* failure — the pipeline honestly cannot proceed, so it raises. A KB timeout during enrichment is *optional* — the memo is still valid, just lower-confidence, so it degrades and reports why. Emitting a `confidence` score turns "some agents failed" into a governable signal: above a threshold you ship with a caveat, below it you route to a human. Silent success on partial data is the actual danger — never let a degraded run masquerade as a clean one.

## 4. Runbook — when the orchestrator itself fails

Retries and breakers handle *agent* failure. When the **orchestrator** dies mid-run, the answer is operational, not code — and having written it down is what an AVP is judged on.

In [4]:
RUNBOOK = {
    "detect":  "alert on: no state transition for >60s, or orchestrator liveness probe fails",
    "contain": "circuit breakers already isolate dead downstreams; stop admitting NEW runs",
    "recover": "restart orchestrator; it RESUMES from last checkpoint (thread_id + Saver) "
               "— in-flight runs continue, they do not restart from zero",
    "verify":  "reconcile: every run in 'working' state either completed or re-queued; "
               "no customer sees a half-written memo",
    "prevent": "idempotent steps (safe to re-run), checkpoints after each node, "
               "dead-letter queue for tasks that fail N times",
}
for phase, action in RUNBOOK.items():
    print(f"{phase.upper():8} {action}")

DETECT   alert on: no state transition for >60s, or orchestrator liveness probe fails
CONTAIN  circuit breakers already isolate dead downstreams; stop admitting NEW runs
RECOVER  restart orchestrator; it RESUMES from last checkpoint (thread_id + Saver) — in-flight runs continue, they do not restart from zero
VERIFY   reconcile: every run in 'working' state either completed or re-queued; no customer sees a half-written memo
PREVENT  idempotent steps (safe to re-run), checkpoints after each node, dead-letter queue for tasks that fail N times


☝ The load-bearing line is **recover**: because every node checkpointed its state (LangGraph `thread_id` + a `Saver`, Day 33), a restarted orchestrator resumes in-flight runs from the last completed node instead of re-charging the customer for four LLM calls. That only works if steps are **idempotent** — re-running "score" must be safe. This is why the checkpointing from Day 33 and the durable store from Day 36 aren't optional niceties: they're what make the recovery step in this runbook actually true rather than aspirational.

## 5. Recap

| Failure mode | Mechanism | Cell |
|---|---|---|
| Transient throttle (429) | retry: exponential backoff **+ full jitter**, capped | 1 |
| Dependency down | circuit breaker: CLOSED→OPEN→HALF_OPEN, fail fast | 2 |
| Retry vs breaker | blip → retry; outage → breaker | 1–2 |
| Optional step fails | degrade honestly + emit `confidence` | 3 |
| Required step fails | raise — cannot proceed | 3 |
| Orchestrator dies | resume from checkpoint; idempotent + DLQ | 4 |

**The mindset:** every external call *will* fail eventually — design for it, make failure *observable*, and never let a degraded run look clean. Tomorrow (Day 38): the Module 5 capstone — a full 4-agent credit-analysis system with a human approval gate, wiring together everything from Days 31–37.